# Module 15 — Logistic Regression Practice Problems
### Titanic Survival Prediction

এই notebook-এ Module 15-এর প্রতিটি topic-এর উপর হাতে-কলমে practice করবে।
প্রতিটি problem-এর নিচে code cell-এ solution লিখো।

**Dataset:** `titanic_data_updated.csv`

---
### Problem 1 — Imports and Data Loading

নিচের সব library import করো:
- `numpy`, `pandas`
- `sklearn` থেকে: `train_test_split`, `SimpleImputer`, `OrdinalEncoder`, `OneHotEncoder`, `LabelEncoder`, `StandardScaler`, `MinMaxScaler`, `Pipeline`, `ColumnTransformer`, `LogisticRegression`

`titanic_data_updated.csv` লোড করো `df` নামে এবং প্রথম 5টি row দেখাও।

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder,MinMaxScaler,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression 

In [2]:
df=pd.read_csv('titanic_data_updated (1).csv')
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


---
### Problem 2 — Feature Engineering

দুটি নতুন column তৈরি করো:

1. `Family_Size` — formula: `SibSp + Parch + 1`
2. `Deck` — `Cabin` column-এর NaN গুলো `"Missing"` দিয়ে fill করো, তারপর প্রথম character নিয়ে `Deck` column তৈরি করো

শেষে `df.sample(5)` দিয়ে result দেখাও।

In [3]:
df['Family_Size']=df['SibSp']+df['Parch']+1
df['Cabin']=df['Cabin'].fillna('Missing')

df['Deck']=df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
600,601,yes,second,"Jacobsohn, Mrs. Sidney Samuel (Amy Frances Chr...",female,24.0,2,1,243847,27.000,Missing,S,4,M
740,741,yes,first,"Hawksford, Mr. Walter James",male,NaN,0,0,16988,30.000,D45,S,1,D
462,463,no,first,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.500,E63,S,1,E
276,277,no,third,"Lindblom, Miss. Augusta Charlotta",female,45.0,0,0,347073,7.750,Missing,S,1,M
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.925,Missing,S,1,M


---
### Problem 3 — X and y Split

`df` থেকে features এবং target আলাদা করো:

- `X` = `Survived` বাদে সব column
- `y` = শুধু `Survived` column

`X.shape` এবং `y.shape` print করো।

In [4]:
# YOUR CODE HERE
x=df.drop('Survived',axis=1)

y=df['Survived']
print(x.shape)
print(y.shape)

(891, 13)
(891,)


---
### Problem 4 — Train-Test Split

`train_test_split` ব্যবহার করে `X_train`, `X_test`, `y_train`, `y_test` তৈরি করো।

Parameters:
- `test_size = 0.2`
- `random_state = 42`
- `stratify = y`

`X_train` এবং `X_test`-এর shape print করো।

In [5]:
# YOUR CODE HERE
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [6]:
X_train.shape

(712, 13)

In [7]:
X_test.shape

(179, 13)

---
### Problem 5 — Outlier Handling

**Age column — Z-score method (rows remove করো):**
- `Z_score = (Age - mean) / std`
- যেসব row-এ `|Z_score| > 3`, সেগুলো `X_train` এবং `y_train` থেকে বাদ দাও

**Fare column — IQR clipping (rows রাখো, values clip করো):**
- `Q1` = 25th percentile, `Q3` = 75th percentile
- `IQR = Q3 - Q1`
- `minimum = max(0, Q1 - 1.5 * IQR)`
- `maximum = Q3 + 1.5 * IQR`
- `clip()` দিয়ে Fare column-কে `[minimum, maximum]` range-এ রাখো

In [8]:
# YOUR CODE HERE
mean_age=df['Age'].mean()
std_age=df['Age'].std()
X_train['Z_score']=(X_train['Age']-mean_age)/mean_age

musk=(abs(X_train['Z_score']<=3))
X_train=X_train[musk]
y_train=y_train[musk]

In [9]:
Q1=X_train['Fare'].quantile(0.25)
Q3=X_train['Fare'].quantile(0.75)

IQR=Q3-Q1
minimum=max(0,Q1-1.5*IQR)
maximum=Q3+1.5*IQR

X_train['Fare']=X_train['Fare'].clip(minimum,maximum)

---
### Problem 6 — Numerical Pipelines

দুটি numerical pipeline তৈরি করো:

**p1** — `Age` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='mean')`
- Step 2: `StandardScaler()`

**p2** — `Fare` এবং `Family_Size` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='median')`
- Step 2: `MinMaxScaler()`

In [10]:
# YOUR CODE HERE
p1=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('scaler',StandardScaler())
    ]
)
p2=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',MinMaxScaler())
    ]
)

---
### Problem 7 — Categorical Pipelines and ColumnTransformer

দুটি categorical pipeline তৈরি করো:

**p3** — `Embarked`, `Sex`, `Deck` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')`

**p4** — `Pclass` column-এর জন্য:
- `categories = [['third', 'second', 'first']]`
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OrdinalEncoder(categories=categories)`
- Step 3: `MinMaxScaler()`

তারপর `ColumnTransformer` দিয়ে `preprocessor` তৈরি করো:
- `pipeline_1` → `p1` → `['Age']`
- `pipeline_2` → `p2` → `['Fare', 'Family_Size']`
- `pipeline_3` → `p3` → `['Embarked', 'Sex', 'Deck']`
- `pipeline_4` → `p4` → `['Pclass']`
- `remainder='drop'`

In [11]:
# YOUR CODE HERE
categories = [['third','second','first']]

In [12]:
p3=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
       
    ]
)
p4=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OrdinalEncoder(categories=categories)),
        ('scaler',MinMaxScaler())
    
    ]
    
)

preprocessor=ColumnTransformer(
    transformers=[
        ('pipeline_1',p1,['Age']),
        ('pipeline_2',p2,['Fare','Family_Size']),
        ('pipeline_3',p3,['Embarked','Sex','Deck']),
        ('pipeline_4',p4,['Pclass'])
      
    ],
    remainder='drop'
)
preprocessor



,transformers,"[('pipeline_1', ...), ('pipeline_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


In [13]:
#preprocessor = ColumnTransformer(
 #   transformers=[
      #  ('Pipeline_1',p1,['Age']),
      #  ('pipeline_2',p2,['Fare','Family_Size']),
       # ('pipeline_3',p3,['Embarked','Sex','Deck']),
       # ('pipeline_4',p4,['Pclass'])
    #],
    #remainder='drop'
#)
#preprocessor

---
### Problem 8 — Target Column Encoding

`y_train` এবং `y_test`-এ এখন `"yes"` এবং `"no"` string আছে। Model train করার আগে এগুলো numeric করতে হবে।

- `LabelEncoder` দিয়ে `y_train` এবং `y_test` encode করো
- encode করার আগে এবং পরে unique values print করো

In [14]:
# YOUR CODE HERE
le= LabelEncoder()

y_train=le.fit_transform(y_train)
y_test=le.fit_transform(y_test)

---
### Problem 9 — Model Training and Prediction

`lr_model` নামে একটি final `Pipeline` তৈরি করো:
- Step 1: `preprocessor` (Problem 7-এ তৈরি করা)
- Step 2: `LogisticRegression(class_weight='balanced', max_iter=1000)`

তারপর:
- `lr_model.fit()` দিয়ে `X_train` এবং `y_train` দিয়ে model train করো
- `predict()` দিয়ে `X_test`-এর prediction করো, `y_pred` নামে save করো
- প্রথম 10টি prediction print করো

In [15]:
# YOUR CODE HERE
lr_model=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',LogisticRegression(class_weight='balanced',max_iter=1000))
    ]
)
lr_model

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline_1', ...), ('pipeline_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
lr_model.fit(X_train,y_train)

lr_model['model'].coef_
lr_model['model'].intercept_
lr_model['model'].classes_

y_pred=lr_model.predict(X_test)
y_prob=lr_model.predict_proba(X_test)



In [22]:
y_prob[:10]

array([[0.85500733, 0.14499267],
       [0.92174397, 0.07825603],
       [0.77369249, 0.22630751],
       [0.92448153, 0.07551847],
       [0.37791038, 0.62208962],
       [0.43632533, 0.56367467],
       [0.24788908, 0.75211092],
       [0.57571888, 0.42428112],
       [0.47321943, 0.52678057],
       [0.77864041, 0.22135959]])

---
### Problem 10 — Evaluation

`sklearn.metrics` থেকে `accuracy_score`, `precision_score`, `recall_score` import করো।

`y_test` এবং `y_pred` দিয়ে তিনটি score calculate করো এবং নিচের format-এ print করো:

```
Accuracy  : 0.xx
Precision : 0.xx
Recall    : 0.xx
```

In [25]:
# YOUR CODE HERE
from sklearn.metrics import accuracy_score,precision_score,recall_score


In [28]:
accuracy=accuracy_score(y_test,y_pred)
print("Accuracy is :",accuracy)
Precision=precision_score(y_test,y_pred)
print("Precision is:",Precision)
Recall=recall_score(y_test,y_pred)
print("Recall is:",Recall)


Accuracy is : 0.7653631284916201
Precision is: 0.6753246753246753
Recall is: 0.7536231884057971
